In [1]:
import ollama
import json

### TOOLS AS A FUNCTION TAAF

In [2]:
from datetime import datetime

def get_user_details():
    return '''
user name is rushal mdhr. users birthday is in jan 1st. User love to code. He created some AIs and want a job at a good company
'''

def create_file(filename,data):
    with open(filename,'w') as f:
        f.write(data)
    return "file created"
        
def get_current_date():
    return datetime.now()

In [3]:
print(get_current_date())

2026-08-29 08:48:53.649554


### CREATE TOOLS

In [4]:
def create_tools(name, description="", type="function", properties=None, required=None):
    return {
        "type": type,
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties or {},  # Pass as dict
                "required": required or []
            }
        }
    }

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": 'get_user_details',
            "description": "Get the user details.",
            "parameters": {
                "type": "object",
                "properties": {
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": 'get_current_date',
            "description": "get the current date",
            "parameters": {
                "type": "object",
                "properties": {
                },
                "required": []
            }
        }
    }
]

In [6]:
tools

[{'type': 'function',
  'function': {'name': 'get_user_details',
   'description': 'Get the user details.',
   'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
 {'type': 'function',
  'function': {'name': 'get_current_date',
   'description': 'get the current date',
   'parameters': {'type': 'object', 'properties': {}, 'required': []}}}]

### whos talking

In [7]:
def user(content):
    return {'role':'user','content':content}
def assistent(content):
    return {'role':'assistent','content':content}
def system(content):
    return {'role':'system','content':content}
def tool_content(content):
    return {'role':'tool','content':content}

### AI

In [8]:

def toai(messages,tools=None):
    return ollama.chat(
        model="llama3.2:latest",
        messages=messages,
        tools=tools,
    )

### MESSAGES

In [9]:
prompt = '''
you are a helpful assistent who can use tools to solve problems of the user , 
u can also help them by telling their birthday time remaining by accessing todays date
also only call tools when need 
'''

In [27]:
messages=[]

In [28]:
messages.append(system(prompt))

In [29]:
messages.append(user("if my birthday in jan and its aug month how much month for my birthday?"))

In [13]:
# messages.append(user("hello! can you tell who am i ?  and how much time there is for my brithday to come comparing to todays date. give me exact number "))

In [30]:
messages

[{'role': 'system',
  'content': '\nyou are a helpful assistent who can use tools to solve problems of the user , \nu can also help them by telling their birthday time remaining by accessing todays date\nalso only call tools when need \n'},
 {'role': 'user',
  'content': 'if my birthday in jan and its aug month how much month for my birthday?'}]

In [16]:
response = toai(messages,tools=tools)

In [17]:
print(response.message.content)

In [18]:
print(response.message.tool_calls)

[ToolCall(function=Function(name='get_current_date', arguments={'date': None, 'year': None, 'month': None, 'day': None}))]


In [19]:
def call_tools(function_name,arguments):
    match function_name:
        case 'get_user_details' : return get_user_details()
        case 'get_current_date' : return get_current_date()
        case 'create_file' : return create_file(**arguments)
        case _: return "unknown function called"
    

In [20]:
if response.message.tool_calls:
    print(f"🧠 Model is requesting to call {len(response.message.tool_calls)} tool(s):")
    # Dictionary to store results
    tool_results = {}
    
    # Process all tool calls (parallel execution)
    for tool in response.message.tool_calls:
        function_name = tool.function.name
        arguments = tool.function.arguments  # Already a dict in recent Ollama
        
        print(f"   🔧 Calling: {function_name} with {arguments}")
        result = call_tools(function_name,arguments)
        tool_results[function_name] = str(result)
        result = json.dumps(tool_results)
    print("\n📊 Tool results:", result)
    messages.append(tool_content(result))

🧠 Model is requesting to call 1 tool(s):
   🔧 Calling: get_current_date with {'date': None, 'year': None, 'month': None, 'day': None}

📊 Tool results: {"get_current_date": "2026-08-29 08:50:11.179375"}


In [31]:
response = toai(messages=messages)

In [32]:
print(response.message.content)

If your birthday is in January and it's August, that means your birthday has passed.

August - July = 1 month
January - December = 12 months

So, if you subtract 12 months from August, you'll get the previous year. That means your birthday was actually in August of the previous year.

As for how many months are left until your birthday, since it's currently August, there are:

0 months left until your birthday this year (since it's already happened)
12 months left until your next birthday


In [ ]:
messages.append(user("yo how can it "))

In [33]:
messages.append(assistent(response.message.content))

In [86]:
messages

[{'role': 'system',
  'content': '\nyou are a helpful assistent who can use tools to solve problems of the user , \nu can also help them by telling their birthday time remaining by accessing todays date\nalso only call tools when need \n'},
 {'role': 'user',
  'content': 'hello! can you tell who am i ?  and how much time there is for my brithday to come'},
 {'role': 'tool',
  'content': '{"get_user_details": "\\nuser name is rushal mdhr. users birthday is in jan 1st. User love to code. He created some AIs and want a job at a good company\\n"}'}]